# <i>CatBoost learning to rank on Microsoft dataset</i>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/catboost/tutorials/blob/master/ranking/ranking_tutorial.ipynb)

In [ ]:
pip install catboost pandas numpy scikit-learn

In [2]:
from catboost import CatBoostRanker, Pool, MetricVisualizer
from copy import deepcopy
import numpy as np
import os
import pandas as pd

### Задача ранжирования

Давайте введем обозначение:
* Пусть $Q = \{q_1, \dots, q_n\}$ - множество групп
* $D_q = \{d_{q1}, \dots, d_{qm}\}$ - набор объектов, извлеченных для группы $q$
* $L_q = \{l_{q1}, \dots, l_{qm}\}$ - метки соответствия для объектов из набора $D_q$

Каждый объект $d_{qi}$ представлен в векторном пространстве признаков, описывающем связи между группой и объектом.

Таким образом, каждая группа связана с набором объектов. Например, группа - это запрос, а объект - документ, если мы ранжируем документы по поисковому запросу.

Цель состоит в том, чтобы изучить функцию ранжирования $f = f(d_{qi})$ таким образом, чтобы ранжирование объектов $d_{qi}$ для всех групп из $Q$ на основе их оценок $x_{qi} = f(d_{qi})$ было следующим максимально приближенный к идеальному рейтингу по мнению редакции $l_{qi}$.

### Ранжирование показателей качества:
* __Precision__
    $$ \mbox{P}=\frac{|\{\mbox{соответствующие документы}\}\cap\{\mbox{восстановленные документы}\}|}{|\{\ mbox{восстановленные документы}\}|} $$
* __Recall__
    $$ \mbox{R}=\frac{|\{\mbox{соответствующие документы}\}\cap\{\mbox{восстановленные документы}\}|}{|\{\ mbox{соответствующие документы}\}|} $$
    
    Обозначение $@k$ означает, что показатель рассчитывается для первых $k$ документов из списка ранжирования.

    Например, если 1,2,5,7,9 - это ранги соответствующих документов (перечисления начинаются с номера 1) из десяти восстановленных, то $P@5$ будет равно $\frac{3}{5}$.

* __Mean average precision (MAP)__
    $$\frac{1}{|Q|}\sum_{q \в Q} \frac{1} _BOS_|\mbox{соответствующие документы в } D_q|} \sum_{k} P@k(q) \times rel(q, k) $$
    
    Где $rel(q, k)$ - это показатель релевантности документа, находящегося на k-й позиции в нашем рейтинге $D_q$. Этот показатель вычисляет среднюю точность для запроса, взвешенного по релевантности документа, а затем вычисляет среднее значение между всеми запросами.
    
* ___Discounted cumulative gain (DCG)__
    $$\sum_{k=1}^{mq} \frac{2 ^ {l_{qk}}} _BOS_\log_2(k+1)}$$
    
    Эта метрика учитывает поведение пользователя: внимание пользователя сначала высоко, а затем нелинейно снижается к концу.
    
* __NDCG__ - нормализованный DCG = DCG $~ / ~ $ IDCG, где IDCG - максимально возможное значение DCG при заданном наборе меток релевантности.

* __AverageGain__ - представляет среднее значение значений меток для объектов с заданными верхними значениями меток.

* __[Найдено](https://tech.yandex.com/catboost/doc/dg/references/pfound-docpage/#pfound)__
    
Подробнее в wiki: https://en.wikipedia.org/wiki/Evaluation_measures_(information_retrieval)

Параметр $@k$ для каждой метрики может быть указан через параметр метрики "top", например, "NDCG:top=10" будет означать NDCG@10.

### Download part of [MSRank](https://www.microsoft.com/en-us/research/project/mslr/) dataset from CatBoost datasets storage

In [3]:
from catboost.datasets import msrank_10k
train_df, test_df = msrank_10k()

X_train = train_df.drop([0, 1], axis=1).values
y_train = train_df[0].values
queries_train = train_df[1].values

X_test = test_df.drop([0, 1], axis=1).values
y_test = test_df[0].values
queries_test = test_df[1].values

### Dataset analysis

__Number of documents__

In [4]:
num_documents = X_train.shape[0]
print(num_documents)

10000


__Number of features__

In [5]:
X_train.shape[1]

136

__Relevance labels statistics__

0 - irrelevant, 1 - highly relevant. Table represents number of documents for each value.

In [6]:
from collections import Counter
Counter(y_train).items()

dict_items([(2.0, 1326), (0.0, 5481), (1.0, 3000), (3.0, 142), (4.0, 51)])

For calculation such metrics as NDCG and PFound relevances should be in segment \[0,1\].

In [7]:
max_relevance = np.max(y_train)
y_train /= max_relevance
y_test /= max_relevance

__Number of queries__

In [8]:
num_queries = np.unique(queries_train).shape[0]
num_queries

87

### Creation of CatBoost pools

In [9]:
train = Pool(
    data=X_train,
    label=y_train,
    group_id=queries_train
)

test = Pool(
    data=X_test,
    label=y_test,
    group_id=queries_test
)

### You can also create pools from files

In [10]:
data_dir = './msrank'

if not os.path.exists(data_dir):
    os.makedirs(data_dir)

train_file = os.path.join(data_dir, 'train.csv')
test_file = os.path.join(data_dir, 'test.csv')

train_df.to_csv(train_file, index=False, header=False)
test_df.to_csv(test_file, index=False, header=False)

In [11]:
description_file = os.path.join(data_dir, 'dataset.cd')
with open(description_file, 'w') as f:
    f.write('0\tLabel\n')
    f.write('1\tQueryId\n')

In [12]:
Pool(data=train_file, column_description=description_file, delimiter=',')

### <span style="color:#ce2029">Attention:</span> all objects in dataset must be grouped by group_id

For example, if the dataset consits of five documents
\[d1, d2, d3, d4, d5\] with corresponding queries \[q1, q2, q2, q1, q2\] then the dataset should be look like:

$$\begin{pmatrix}
    d_1, q_1, f_1\\
    d_4, q_1, f_4\\
    d_2, q_2, f_2\\
    d_3, q_2, f_3\\
    d_5, q_2, f_5\\
\end{pmatrix} \hspace{6px} \texttt{or} \hspace{6px}
\begin{pmatrix}
    d_2, q_2, f_2\\
    d_3, q_2, f_3\\
    d_5, q_2, f_5\\
    d_1, q_1, f_1\\
    d_4, q_1, f_4\\
\end{pmatrix}$$

where $f_i$ is feature vector of i-th document.

### Reducing problem to machine learning task

The first and simplest idea is to try predicting document relevance $l_q$ minimizing RMSE.

$$\frac{1}{N}\sqrt{ \sum_q \sum_{d_{qk}} \left(f(d_{qk}) - l_{qk} \right)^2 }$$

In [ ]:
default_parameters = {
    'iterations': 2000,
    'custom_metric': ['NDCG', 'PFound', 'AverageGain:top=10'],
    'verbose': False,
    'random_seed': 0,
}

parameters = {}

In [ ]:
def fit_model(loss_function, additional_params=None, train_pool=train, test_pool=test):
    parameters = deepcopy(default_parameters)
    parameters['loss_function'] = loss_function
    parameters['train_dir'] = loss_function

    if additional_params is not None:
        parameters.update(additional_params)

    model = CatBoostRanker(**parameters)
    model.fit(train_pool, eval_set=test_pool, plot=True)

    return model

Lets train the simplest model and also demonstrate precision/recall metrics from introduction.

In [ ]:
model = fit_model('RMSE', {'custom_metric': ['PrecisionAt:top=10', 'RecallAt:top=10', 'MAP:top=10']})

### Group weights parameter
Suppose we know that some queries are more important than others for us.<br/>
The word "importance" used here in terms of accuracy or quality of CatBoostRanker prediction for given queries.<br/>
You can pass this additional information for learner using a ``group_weights`` parameter.<br/>
Under the hood, CatBoostRanker uses this weights in loss function simply multiplying it on a group summand.<br/>
So the bigger weight $\rightarrow$ the more attention for query.<br/>
Let's show an example of training procedure with random query weights.

In [ ]:
def create_weights(queries):
    query_set = np.unique(queries)
    query_weights = np.random.uniform(size=query_set.shape[0])
    weights = np.zeros(shape=queries.shape)

    for i, query_id in enumerate(query_set):
        weights[queries == query_id] = query_weights[i]

    return weights


train_with_weights = Pool(
    data=X_train,
    label=y_train,
    group_weight=create_weights(queries_train),
    group_id=queries_train
)

test_with_weights = Pool(
    data=X_test,
    label=y_test,
    group_weight=create_weights(queries_test),
    group_id=queries_test
)

fit_model(
    'RMSE',
    additional_params={'train_dir': 'RMSE_weigths'},
    train_pool=train_with_weights,
    test_pool=test_with_weights
)

### A special case: top-1 prediction

Someday you may face with a problem $-$ you will need to predict the top one most relevant object for a given query.<br/>
For this purpose CatBoostRanker has a mode called __QuerySoftMax__.

Suppose our dataset contain a binary target: 1 $-$ mean best document for a query, 0 $-$ others.<br/>
We will maximize the probability of being the best document for given query.<br/>
MSRANK dataset doesn't contain binary labels, but for example of method __QuerySoftMax__ we convert it to that format,<br/> choosing a best document for every query.

In [ ]:
def get_best_documents(labels, queries):
    query_set = np.unique(queries)
    num_queries = query_set.shape[0]
    by_query_arg_max = {query: -1 for query in query_set}

    for i, query in enumerate(queries):
        best_idx = by_query_arg_max[query]
        if best_idx == -1 or labels[best_idx] < labels[i]:
            by_query_arg_max[query] = i

    binary_best_docs = np.zeros(shape=labels.shape)
    for arg_max in by_query_arg_max.values():
        binary_best_docs[arg_max] = 1.

    return binary_best_docs

In [ ]:
best_docs_train = get_best_documents(y_train, queries_train)
best_docs_test = get_best_documents(y_test, queries_test)

train_with_weights = Pool(
    data=X_train,
    label=best_docs_train,
    group_id=queries_train,
    group_weight=create_weights(queries_train)
)

test_with_weights = Pool(
    data=X_test,
    label=best_docs_test,
    group_id=queries_test,
    group_weight=create_weights(queries_test)
)

fit_model(
    'QuerySoftMax',
    additional_params={'custom_metric': 'AverageGain:top=1'},
    train_pool=train_with_weights,
    test_pool=test_with_weights
)

### Reducing ploblem, step 2

Now lets look at example of documents relevance:

$$
    \begin{align}
    labels(q_1) &= \begin{bmatrix}
           4 \\
           3 \\
           3 \\
           1
         \end{bmatrix},
    labels(q_2) &= \begin{bmatrix}
           2 \\
           1 \\
           1 \\
           0
         \end{bmatrix}
   \end{align}
$$

This means that with RMSE loss function we pay more attention to q1 than q2.

To avoid this problem we introduce into RMSE a coefficient $c_q$ which depends only on query (and if fact equals to the mean of the difference between prediction and label).

$$\frac{1}{N}\sqrt{ \sum_q \sum_{d_{qk}} \left(f(d_{qk}) - l_{qk} - \color{red}{c_{q}} \right)^2 }$$

In [ ]:
fit_model('QueryRMSE')

### Reducing problem, step 3

Since the goal of ranking is to predict a list of documents (which can be generated from given document relevances) RMSE loss function doesn't take into account relations between documents: the first is better than second, second is better than third and fifth etc.

We can easily bring this information into the loss function, reducing problem not to regression but classification for two documents $(d_i, d_j)$ -- does $i$th better than $j$th or not.

So we minimize the negative loglikelihood:

$$ - \sum_{i,j \in Pairs} \log \left( \frac{1}{1 + \exp{-(f(d_i) - f(d_j))}} \right) $$

Methods based on pair comparisons called __pairwise__ in CatBoostRanker this objective called __PairLogit__.

There's no need to change the dataset CatBoost generate the pairs for us. The number of generating pairs managed via parameter max_size.

In [ ]:
fit_model('PairLogit')

Also we can to specify the pairs directly. There are two ways to do that:

1. Two-dimensional matrix with shape=(num_pairs, 2) $\rightarrow$ (winner_id, loser_id): list, numpy.ndarray, pandas.DataFrame.
2. Path two the input file that contains pair descriptions:
    * Row format: $\texttt{[winner index, loser index, pair weight]}$

In [ ]:
def read_groups(file_name):
    groups = {}
    group_ids = []

    with open(file_name) as f:
        for doc_id, line in enumerate(f):
            line = line.split(',')[:2]

            label, query_id = float(line[0]), int(line[1])
            if query_id not in groups:
                groups[query_id] = []
            groups[query_id].append((doc_id, label))

            group_ids.append(query_id)

    return groups, group_ids

train_groups, train_group_ids = read_groups(train_file)
assert num_queries == len(train_groups)

In [ ]:
pairs = []

for group in train_groups.values():
    for i in range(len(group)):
        for j in range(i, len(group)):
            if i == j:
                continue
            doc_i, relevance_i = group[i]
            doc_j, relevance_j = group[j]
            if relevance_i < relevance_j:
                pairs.append((doc_j, doc_i))
            else:
                pairs.append((doc_i, doc_j))

pairs_file = os.path.join(data_dir, 'pairs.csv')

with open(pairs_file, 'w') as f:
    for pair in pairs:
        f.write(str(pair[0]) + '\t' + str(pair[1]) + '\t1\n')

In [ ]:
pool1 = Pool(data=X_train, label=y_train, group_id=train_group_ids, pairs=pairs)
pool2 = Pool(data=train_file, column_description=description_file, pairs=pairs_file, delimiter=',')

### Reducing problem, step 3.1

Thus we know that $f(d_{qk})$ is a ensemble of trees, we can accurately solve the minimization task from step 3.

This method called __PairLogitPairwise__.

In [ ]:
fit_model('PairLogitPairwise')

### Reducing problem, step 4

Previous loss function directly minimize the number of pairs $(d_i, d_j)$ where $l_i > l_j$ but $f(d_i) < f(d_j)$, simply said the number of incorrectly placed documents.

Since the user attention is high on the first documents and low on last the incorrect switch of the first two documents and last two has different cost.

In steps 3 and 3.1 user can set the weight for pair.

Method __YetiRank__ take this effect into account and generates weights for pairs according to their positions ([paper](https://cache-mskstoredata08.cdn.yandex.net/download.yandex.ru/company/to_rank_challenge_with_yetirank.pdf)).

$$ - \sum_{i,j \in Pairs} \color{red}{w_{ij}} \log \left( \frac{1}{1 + \exp{-(f(d_i) - f(d_j))}} \right) $$

In [ ]:
fit_model('YetiRank')

### Step 4.1

As in step 3.1 __YetiRankPairwise__ is slower than __YetiRank__, but gives more accurate results.

In [ ]:
fit_model('YetiRankPairwise')

In [ ]:
widget = MetricVisualizer(['RMSE', 'QueryRMSE', 'PairLogit', 'PairLogitPairwise', 'YetiRank', 'YetiRankPairwise'])
widget.start()

### Simple classification

Very fast $\rightarrow$ very slow; Simple method $\rightarrow$ complex method; Low quality $\rightarrow$ high quality.

1. RMSE
2. QueryRMSE
3. PairLogit
4. PairLogitPairwise
5. YetiRank
6. YetiRankPairwise

Besides our classification, the quality of the concrete method may depend on your dataset.

Look on NDCG metric of method YetiRank $-$ it's underfitted.

In [ ]:
fit_model('YetiRank', {'train_dir': 'YetiRank-lr-0.3', 'learning_rate': 0.3})

In [ ]:
widget = MetricVisualizer(['YetiRank', 'YetiRank-lr-0.3'])
widget.start()

### Additional parameters

__Metric period__

Period in iterations of calculation metrics. This parameter can speed up training process.

In [ ]:
fit_model('YetiRank', {'metric_period': 50})

__Task type__

You can significantly speed up training procedure by switching to GPU.

In [ ]:
fit_model('YetiRank', {'task_type': 'GPU'})